# yt-dlp server blocking test — no cookies

Use this notebook in Google Colab to reproduce the cloud/server behavior without cookies. It intentionally does **not** authenticate to YouTube.

Expected result: this may fail with a YouTube bot/sign-in challenge on Colab or other server-like IPs. If it succeeds, that only proves the current runtime IP is temporarily allowed.

In [ ]:
!python -m pip install -q -U "yt-dlp[default]" requests
!curl -fsSL https://deno.land/install.sh | sh >/dev/null


In [ ]:
import json
import os
import platform
import shlex
import shutil
import subprocess
from pathlib import Path

import requests

os.environ["PATH"] = "/root/.deno/bin:" + os.environ.get("PATH", "")

CHANNEL_URL = "https://www.youtube.com/@zackdfilms/videos"
PLAYLIST_ENDS = [1, 3, 10, 25]
DOWNLOAD_DIR = Path("/content/yt_dlp_baseline_downloads")
DOWNLOAD_DIR.mkdir(parents=True, exist_ok=True)
COMMON_EJS_ARGS = ["--js-runtimes", "deno"]

def normalize_channel_url(url: str) -> str:
    url = url.strip()
    if "youtube.com/@" in url and not url.rstrip("/").endswith("/videos"):
        return url.rstrip("/") + "/videos"
    return url

def classify_output(output: str, returncode: int) -> str:
    lower = output.lower()
    bot_markers = [
        "sign in to confirm",
        "not a bot",
        "confirm you're not a bot",
        "confirm you?re not a bot",
        "unusual traffic",
    ]
    if any(marker in lower for marker in bot_markers):
        return "BLOCKED_BY_YOUTUBE_BOT_CHECK"
    if "n challenge solving failed" in lower:
        return "EJS_CHALLENGE_SOLVER_FAILED"
    if "only images are available" in lower or "requested format is not available" in lower:
        return "NO_AUDIO_VIDEO_FORMATS"
    if "private video" in lower or "unavailable" in lower:
        return "VIDEO_UNAVAILABLE_OR_PRIVATE"
    if returncode == 0:
        return "SUCCESS"
    return f"FAILED_EXIT_{returncode}"

def run_cmd(label: str, args: list[str]) -> tuple[int, str, str]:
    print("=" * 90)
    print(label)
    print("$", " ".join(shlex.quote(str(arg)) for arg in args))
    proc = subprocess.run(args, text=True, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, env=os.environ.copy())
    output = proc.stdout or ""
    print(output)
    verdict = classify_output(output, proc.returncode)
    print("VERDICT:", verdict)
    return proc.returncode, output, verdict

if shutil.which("ffmpeg") is None:
    subprocess.run(["apt-get", "update", "-qq"], check=False)
    subprocess.run(["apt-get", "install", "-y", "-qq", "ffmpeg"], check=False)


In [ ]:
print("Python:", platform.python_version())
print("Platform:", platform.platform())
try:
    print("Public IP:", requests.get("https://api.ipify.org?format=json", timeout=10).json())
except Exception as exc:
    print("Public IP lookup failed:", exc)

run_cmd("yt-dlp version", ["yt-dlp", "--version"])
run_cmd("Deno version", ["deno", "--version"])
run_cmd("ffmpeg version", ["ffmpeg", "-version"])


## Test 1 — channel listing without cookies

This tests whether cloud IPs can list the first three channel videos without authentication.

In [ ]:
channel_url = normalize_channel_url(CHANNEL_URL)
listing_results = []
for playlist_end in PLAYLIST_ENDS:
    listing_results.append(run_cmd(
        f"Channel flat playlist extraction without cookies, playlist-end={playlist_end}",
        [
            "yt-dlp",
            *COMMON_EJS_ARGS,
            "--flat-playlist",
            "--playlist-end", str(playlist_end),
            "--print", "%(id)s | %(title)s",
            channel_url,
        ],
    ))


## Test 2 — one audio extraction without cookies

This is the closer test for the Whisper fallback path because it actually downloads audio.

In [ ]:
audio_result = run_cmd(
    "One audio extraction without cookies",
    [
        "yt-dlp",
        *COMMON_EJS_ARGS,
        "--playlist-end", "1",
        "-x",
        "--audio-format", "mp3",
        "--audio-quality", "64K",
        "--paths", str(DOWNLOAD_DIR),
        "--output", "%(id)s.%(ext)s",
        channel_url,
    ],
)


In [ ]:
print("Summary")
for playlist_end, result in zip(PLAYLIST_ENDS, listing_results):
    print(f"- Listing {playlist_end}:", result[2])
print("- Audio:", audio_result[2])
print("Downloaded files:", [p.name for p in DOWNLOAD_DIR.glob("*")])

all_results = listing_results + [audio_result]
if any(result[2] == "BLOCKED_BY_YOUTUBE_BOT_CHECK" for result in all_results):
    print("Cloud/server blocking reproduced. Run the cookie notebook next.")
elif any(result[2] == "EJS_CHALLENGE_SOLVER_FAILED" for result in all_results):
    print("The blocker is EJS/challenge solving, not cookies. Check Deno + yt-dlp[default] install output.")
else:
    print("No bot-check was reproduced in this runtime. Re-test on the target GPU/server provider before deciding.")
